In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Hotel Booking Analysis") \
    .getOrCreate()

df = spark.read.csv(
    "hdfs://localhost:9000/user/hotel/data/hotel_data_cleaned.csv",
    header=True,
    inferSchema=True
)

df.createOrReplaceTempView("hotel_bookings")

In [2]:
query_9 = """
WITH waiting_stats AS (
    SELECT
        is_canceled,
        CASE
            WHEN is_canceled = 0 THEN 'Dat phong thanh cong'
            ELSE 'Da huy don'
        END AS booking_status,
        days_in_waiting_list
    FROM hotel_bookings
),
summary AS (
    SELECT
        booking_status,
        COUNT(*) AS total_bookings,
        SUM(CASE WHEN days_in_waiting_list > 0 THEN 1 ELSE 0 END) AS had_waiting,
        ROUND(AVG(days_in_waiting_list), 2) AS avg_waiting_days,
        ROUND(AVG(CASE WHEN days_in_waiting_list > 0
                       THEN days_in_waiting_list END), 2) AS avg_waiting_days_nonzero,
        MAX(days_in_waiting_list) AS max_waiting_days,
        ROUND(SUM(CASE WHEN days_in_waiting_list > 0 THEN 1 ELSE 0 END)
              * 100.0 / COUNT(*), 2) AS pct_had_to_wait
    FROM waiting_stats
    GROUP BY booking_status, is_canceled
)
SELECT
    booking_status,
    total_bookings,
    had_waiting,
    pct_had_to_wait,
    avg_waiting_days,
    avg_waiting_days_nonzero,
    max_waiting_days
FROM summary
ORDER BY booking_status DESC
"""

result_9 = spark.sql(query_9)
result_9.show(truncate=False)

+--------------------+--------------+-----------+---------------+----------------+------------------------+----------------+
|booking_status      |total_bookings|had_waiting|pct_had_to_wait|avg_waiting_days|avg_waiting_days_nonzero|max_waiting_days|
+--------------------+--------------+-----------+---------------+----------------+------------------------+----------------+
|Dat phong thanh cong|75166         |1339       |1.78           |1.59            |89.25                   |379             |
|Da huy don          |44224         |2359       |5.33           |3.56            |66.82                   |391             |
+--------------------+--------------+-----------+---------------+----------------+------------------------+----------------+



In [3]:
query_10 = """
WITH guest_classified AS (
    SELECT
        is_canceled,
        adults,
        babies,
        stays_in_weekend_nights,
        stays_in_week_nights,
        hotel,
        adr,
        CASE
            WHEN (CAST(children AS INTEGER) > 0 OR babies > 0)
                THEN 'Gia dinh'
            WHEN (adults = 1 AND CAST(children AS INTEGER) = 0 AND babies = 0)
                THEN 'Don le'
            ELSE 'Nhom ban'
        END AS guest_group
    FROM hotel_bookings
    WHERE adults > 0
)
SELECT
    guest_group,
    hotel,
    COUNT(*) AS total_bookings,
    ROUND(AVG(stays_in_weekend_nights + stays_in_week_nights), 2) AS avg_total_nights,
    ROUND(AVG(CASE WHEN is_canceled = 0
                   THEN stays_in_weekend_nights + stays_in_week_nights END), 2)
        AS avg_nights_stayed,
    SUM(is_canceled) AS total_canceled,
    ROUND(SUM(is_canceled) * 100.0 / COUNT(*), 2) AS cancel_rate_pct,
    ROUND(AVG(adr), 2) AS avg_adr
FROM guest_classified
GROUP BY guest_group, hotel
ORDER BY guest_group, hotel
"""

result_10 = spark.sql(query_10)
result_10.show(truncate=False)

+-----------+------------+--------------+----------------+-----------------+--------------+---------------+-------+
|guest_group|hotel       |total_bookings|avg_total_nights|avg_nights_stayed|total_canceled|cancel_rate_pct|avg_adr|
+-----------+------------+--------------+----------------+-----------------+--------------+---------------+-------+
|Don le     |City Hotel  |15564         |2.53            |2.31             |5372          |34.52          |93.52  |
|Don le     |Resort Hotel|7013          |3.0             |2.8              |1183          |16.87          |54.99  |
|Gia dinh   |City Hotel  |5180          |3.33            |3.09             |1778          |34.32          |152.5  |
|Gia dinh   |Resort Hotel|3929          |4.75            |4.41             |1397          |35.56          |161.14 |
|Nhom ban   |City Hotel  |58196         |3.06            |3.09             |25845         |44.41          |104.62 |
|Nhom ban   |Resort Hotel|29105         |4.58            |4.49          